# S4 · AndinaLog 03B · Notebook 1 · Diagnóstico de flota

Este notebook diagnostica el Bronze oficial `andinalog_flota.csv`. Conserva sus cuatro columnas originales y todas las filas; no normaliza identificadores, no elimina duplicados y no altera capacidades. El tratamiento corresponde al notebook 2, después de aprobar sus reglas.

Cada ejecución genera cuatro CSV bajo `proyecto-integrador/andinalog_flota/notebook1/salidas/`: diagnosticado, problemas, cuarentena y reporte de calidad con SHA-256 del Bronze.


## 1 · Configuración y origen

En local, ejecuta dentro de `practicasNotebookColab`. En Colab, ajusta `RUTA_PROYECTO_DRIVE` a la carpeta que contiene `datasets/` y `proyecto-integrador/`.


In [2]:
from pathlib import Path
import hashlib
import os
import tempfile
import pandas as pd

ENTORNO = "auto"  # auto, local o drive
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
CARPETA_DATASETS = "AndinaLog_03B_Bronce"
NOMBRE_CSV = "andinalog_flota.csv"
VERSION_DIAGNOSTICO = "GIAD-M3-S4-FLOTA-diagnostico-v2"
COLUMNAS_ORIGINALES = ["camion_id", "centro_distribucion_base", "capacidad_kg", "tipo_camion"]

def encontrar_raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if ((carpeta / "datasets" / CARPETA_DATASETS).is_dir()
                and (carpeta / "proyecto-integrador").is_dir()):
            return carpeta
    raise FileNotFoundError("Ejecuta el notebook dentro de practicasNotebookColab")

def configurar_rutas(entorno, ruta_drive):
    if entorno == "auto":
        entorno = "drive" if "google.colab" in __import__("sys").modules else "local"
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(ruta_drive)
    elif entorno == "local":
        raiz = encontrar_raiz_local()
    else:
        raise ValueError("ENTORNO debe ser auto, local o drive")
    bronze = raiz / "datasets" / CARPETA_DATASETS / NOMBRE_CSV
    salidas = raiz / "proyecto-integrador" / "andinalog_flota" / "notebook1" / "salidas"
    if not bronze.is_file():
        raise FileNotFoundError(f"No se encontró el CSV Bronze: {bronze}")
    return bronze, salidas

RUTA_BRONZE, DIRECTORIO_SALIDAS = configurar_rutas(ENTORNO, RUTA_PROYECTO_DRIVE)
print("Bronze:", RUTA_BRONZE)
print("Salidas:", DIRECTORIO_SALIDAS)


Bronze: c:\Users\remrodri\Github\practicasNotebookColab\datasets\AndinaLog_03B_Bronce\andinalog_flota.csv
Salidas: c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_flota\notebook1\salidas


## 2 · Carga y contrato

El Bronze se lee como texto, incluidos los vacíos. Se exige el esquema de cuatro columnas en el orden original; las conversiones numéricas usadas para diagnóstico no se exportan como cambios.


In [3]:
def cargar_bronze(ruta):
    huella = hashlib.sha256(ruta.read_bytes()).hexdigest()
    df = pd.read_csv(ruta, dtype="string", encoding="utf-8-sig", keep_default_na=False)
    return df, huella

def validar_esquema(df):
    if list(df.columns) != COLUMNAS_ORIGINALES:
        faltantes = sorted(set(COLUMNAS_ORIGINALES) - set(df.columns))
        extras = sorted(set(df.columns) - set(COLUMNAS_ORIGINALES))
        raise ValueError(f"Esquema inesperado. Faltantes: {faltantes}; extras: {extras}; orden: {list(df.columns)}")
    if not df.columns.is_unique:
        raise ValueError("Nombres de columnas duplicados")
    return df

df_bronze, HASH_BRONZE = cargar_bronze(RUTA_BRONZE)
validar_esquema(df_bronze)
print(f"Bronze: {len(df_bronze):,} filas × {len(df_bronze.columns)} columnas")
print("SHA-256:", HASH_BRONZE)
display(df_bronze.head())


Bronze: 32 filas × 4 columnas
SHA-256: 5da0620ea1883c3bad57ba31f45c24b454fe3dadede3cf2370eff459714fbf14


,camion_id,centro_distribucion_base,capacidad_kg,tipo_camion
0,CAM-01,Cochabamba,5000,Refrigerado
1,CAM-02,Cochabamba,5000,Refrigerado
2,CAM-03,Oruro,12000,Seco
3,CAM-04,La Paz,2000,Seco
4,CAM-05,Cochabamba,8000,Seco


## 3 · Catálogo y reglas de diagnóstico

Se detectan IDs vacíos o distintos de `CAM-##`, centros vacíos, capacidades faltantes, no numéricas o no positivas, y tipos distintos de `Refrigerado` o `Seco`. Las copias completas se distinguen de claves `camion_id` con datos contradictorios. Una coincidencia que surgiría al quitar espacios y pasar a mayúsculas se registra por separado.

El diagnóstico no elige un camión canónico ni corrige las variantes de escritura. Tampoco impone un catálogo de centros o un máximo de capacidad sin una regla de negocio validada.


In [4]:
CATALOGO_PROBLEMAS = pd.DataFrame([
    ("camion_id", "FALTANTE", "Identificador vacío"),
    ("camion_id", "FORMATO_INVALIDO", "No cumple CAM-## exactamente"),
    ("camion_id", "DUPLICADO_IDENTICO", "Copia posterior con los cuatro campos iguales"),
    ("camion_id", "ID_EN_CONFLICTO", "Mismo ID exacto con otros campos diferentes; se marcan todas las filas"),
    ("camion_id", "COLISION_ID_NORMALIZADO", "Coincidencia al quitar espacios y usar mayúsculas"),
    ("centro_distribucion_base", "FALTANTE", "Centro vacío"),
    ("capacidad_kg", "FALTANTE", "Capacidad vacía"),
    ("capacidad_kg", "NO_NUMERICA", "Capacidad no convertible a número"),
    ("capacidad_kg", "NO_POSITIVA", "Capacidad menor o igual a cero"),
    ("tipo_camion", "FALTANTE", "Tipo vacío"),
    ("tipo_camion", "TIPO_NO_RECONOCIDO", "Distinto de Refrigerado o Seco"),
], columns=["columna_afectada", "codigo_error", "criterio"])
display(CATALOGO_PROBLEMAS)

def registrar_problema(df, mascara, columna, codigo):
    mascara = mascara.fillna(False).astype(bool)
    filas = df.loc[mascara, ["fila_bronze"]].copy()
    filas["columna_afectada"] = columna
    filas["codigo_error"] = codigo
    filas["valor_original"] = df.loc[mascara, columna].astype("string").to_numpy()
    return filas

def diagnosticar(df_bronze):
    principal = df_bronze.copy(deep=True)
    principal.insert(0, "fila_bronze", range(1, len(principal) + 1))
    cid = principal["camion_id"].astype("string")
    cid_limpio = cid.str.strip()
    cid_normal = cid_limpio.str.upper()
    # Dentro de una clave exacta, distinguir copia completa y contenido contradictorio.
    firmas = principal[COLUMNAS_ORIGINALES].astype("string").agg("\x1f".join, axis=1)
    variantes = firmas.groupby(cid_limpio, dropna=False).transform("nunique")
    conflicto = cid_limpio.ne("") & cid_limpio.duplicated(keep=False) & variantes.gt(1)
    identico = principal[COLUMNAS_ORIGINALES].duplicated(keep="first") & ~conflicto
    colision = (cid_normal.ne("") & cid_normal.duplicated(keep="first")
                & ~cid_limpio.duplicated(keep="first"))
    centro = principal["centro_distribucion_base"].astype("string").str.strip()
    capacidad = principal["capacidad_kg"].astype("string").str.strip()
    capacidad_num = pd.to_numeric(capacidad, errors="coerce")
    tipo = principal["tipo_camion"].astype("string").str.strip()
    hallazgos = [
        registrar_problema(principal, cid_limpio.eq(""), "camion_id", "FALTANTE"),
        registrar_problema(principal, cid.ne("") & ~cid.str.fullmatch(r"CAM-\d{2}").fillna(False), "camion_id", "FORMATO_INVALIDO"),
        registrar_problema(principal, identico, "camion_id", "DUPLICADO_IDENTICO"),
        registrar_problema(principal, conflicto, "camion_id", "ID_EN_CONFLICTO"),
        registrar_problema(principal, colision, "camion_id", "COLISION_ID_NORMALIZADO"),
        registrar_problema(principal, centro.eq(""), "centro_distribucion_base", "FALTANTE"),
        registrar_problema(principal, capacidad.eq(""), "capacidad_kg", "FALTANTE"),
        registrar_problema(principal, capacidad.ne("") & capacidad_num.isna(), "capacidad_kg", "NO_NUMERICA"),
        registrar_problema(principal, capacidad_num.notna() & capacidad_num.le(0), "capacidad_kg", "NO_POSITIVA"),
        registrar_problema(principal, tipo.eq(""), "tipo_camion", "FALTANTE"),
        registrar_problema(principal, tipo.ne("") & ~tipo.isin(["Refrigerado", "Seco"]), "tipo_camion", "TIPO_NO_RECONOCIDO"),
    ]
    problemas = pd.concat(hallazgos, ignore_index=True)
    problemas = problemas.sort_values(["fila_bronze", "columna_afectada", "codigo_error"], kind="stable").reset_index(drop=True)
    problemas["version_diagnostico"] = VERSION_DIAGNOSTICO
    columnas_por_fila = problemas.groupby("fila_bronze")["columna_afectada"].agg(
        lambda valores: "|".join(dict.fromkeys(valores)))
    principal["columnas_con_problemas"] = principal["fila_bronze"].map(columnas_por_fila).fillna("")
    principal["en_cuarentena"] = principal["columnas_con_problemas"].ne("")
    return principal, problemas

df_diagnosticado, df_problemas = diagnosticar(df_bronze)
df_cuarentena = df_diagnosticado.loc[df_diagnosticado["en_cuarentena"]].copy()
print(f"Filas: {len(df_diagnosticado):,}; problemas: {len(df_problemas):,}; cuarentena: {len(df_cuarentena):,}")
display(df_problemas.groupby(["columna_afectada", "codigo_error"]).size().rename("filas").reset_index())


,columna_afectada,codigo_error,criterio
0,camion_id,FALTANTE,Identificador vacío
1,camion_id,FORMATO_INVALIDO,No cumple CAM-## exactamente
2,camion_id,DUPLICADO_IDENTICO,Copia posterior con los cuatro campos iguales
3,camion_id,ID_EN_CONFLICTO,Mismo ID exacto con otros campos diferentes; s...
4,camion_id,COLISION_ID_NORMALIZADO,Coincidencia al quitar espacios y usar mayúsculas
5,centro_distribucion_base,FALTANTE,Centro vacío
6,capacidad_kg,FALTANTE,Capacidad vacía
7,capacidad_kg,NO_NUMERICA,Capacidad no convertible a número
8,capacidad_kg,NO_POSITIVA,Capacidad menor o igual a cero
9,tipo_camion,FALTANTE,Tipo vacío


Filas: 32; problemas: 4; cuarentena: 4


,columna_afectada,codigo_error,filas
0,camion_id,DUPLICADO_IDENTICO,2
1,camion_id,FORMATO_INVALIDO,2


## 4 · Reporte y comprobaciones

El reporte incluye la huella del Bronze y los conteos de hallazgos. Se valida que todos los valores originales permanezcan iguales y que las filas en cuarentena coincidan exactamente con las filas que tienen problemas.


In [5]:
def construir_reporte(df_bronze, principal, problemas, ruta, huella):
    conteos = problemas.groupby(["columna_afectada", "codigo_error"]).size()
    datos = [
        ("archivo_bronze", ruta.name), ("sha256_bronze", huella),
        ("version_diagnostico", VERSION_DIAGNOSTICO),
        ("filas_bronze", len(df_bronze)), ("filas_diagnosticadas", len(principal)),
        ("filas_en_cuarentena", int(principal["en_cuarentena"].sum())),
        ("filas_sin_cuarentena", int((~principal["en_cuarentena"]).sum())),
        ("problemas_detectados", len(problemas)),
    ]
    datos += [(f"{col}:{codigo}", int(total)) for (col, codigo), total in conteos.items()]
    return pd.DataFrame(datos, columns=["metrica", "valor"])

def validar_resultados(df_bronze, principal, problemas, cuarentena, reporte):
    assert list(principal.columns) == ["fila_bronze", *COLUMNAS_ORIGINALES, "columnas_con_problemas", "en_cuarentena"]
    pd.testing.assert_frame_equal(principal[COLUMNAS_ORIGINALES], df_bronze[COLUMNAS_ORIGINALES])
    assert len(principal) == len(df_bronze)
    assert principal["fila_bronze"].is_unique
    assert len(cuarentena) == int(principal["en_cuarentena"].sum())
    assert problemas["fila_bronze"].isin(principal["fila_bronze"]).all()
    assert set(problemas["fila_bronze"]) == set(cuarentena["fila_bronze"])
    pares_catalogo = set(CATALOGO_PROBLEMAS[["columna_afectada", "codigo_error"]].apply(tuple, axis=1))
    pares_problemas = set(problemas[["columna_afectada", "codigo_error"]].apply(tuple, axis=1))
    assert pares_problemas.issubset(pares_catalogo)
    assert len(reporte) >= 8

reporte_calidad = construir_reporte(df_bronze, df_diagnosticado, df_problemas, RUTA_BRONZE, HASH_BRONZE)
validar_resultados(df_bronze, df_diagnosticado, df_problemas, df_cuarentena, reporte_calidad)
display(reporte_calidad)
print("Comprobaciones previas a la exportación: correctas")


,metrica,valor
0,archivo_bronze,andinalog_flota.csv
1,sha256_bronze,5da0620ea1883c3bad57ba31f45c24b454fe3dadede3cf...
2,version_diagnostico,GIAD-M3-S4-FLOTA-diagnostico-v2
3,filas_bronze,32
4,filas_diagnosticadas,32
5,filas_en_cuarentena,4
6,filas_sin_cuarentena,28
7,problemas_detectados,4
8,camion_id:DUPLICADO_IDENTICO,2
9,camion_id:FORMATO_INVALIDO,2


Comprobaciones previas a la exportación: correctas


## 5 · Exportación reproducible

La última celda escribe archivos temporales y luego reemplaza las cuatro salidas en `proyecto-integrador/andinalog_flota/notebook1/salidas/`. Antes de exportar comprueba que el CSV Bronze no cambió.


In [6]:
def exportar_salidas(directorio, tablas, ruta_bronze, huella_inicial):
    if hashlib.sha256(ruta_bronze.read_bytes()).hexdigest() != huella_inicial:
        raise RuntimeError("El CSV Bronze cambió durante la ejecución; no se exportarán resultados")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_flota_",
                                             dir=directorio, encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

tablas_salida = {
    "andinalog_flota_diagnosticado.csv": df_diagnosticado,
    "andinalog_flota_problemas.csv": df_problemas,
    "andinalog_flota_cuarentena.csv": df_cuarentena,
    "andinalog_flota_reporte_calidad.csv": reporte_calidad,
}
for ruta in exportar_salidas(DIRECTORIO_SALIDAS, tablas_salida, RUTA_BRONZE, HASH_BRONZE):
    print(ruta)
print("Bronze intacta; salidas anteriores reemplazadas")


c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_flota\notebook1\salidas\andinalog_flota_diagnosticado.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_flota\notebook1\salidas\andinalog_flota_problemas.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_flota\notebook1\salidas\andinalog_flota_cuarentena.csv
c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\andinalog_flota\notebook1\salidas\andinalog_flota_reporte_calidad.csv
Bronze intacta; salidas anteriores reemplazadas


## Siguiente etapa

El notebook 2 deberá leer el diagnosticado y el detalle de problemas. Los identificadores mal formados y las copias solo se tratarán según reglas aprobadas y verificables.
